In [1]:
import pandas as pd
import numpy as np

# Load field soil moisture data
df = pd.read_csv(r"data\sm_23_04_zhurucay.csv", parse_dates=["TimeStamp"], dayfirst=True)

# Filter to measurements from 23/04/2026
df_day = df[df["TimeStamp"].dt.date == pd.Timestamp("2026-04-23").date()].copy()

print(f"Total records on 23/04/2026: {len(df_day)}")
print(df_day[["TimeStamp", "Latitude", "Longitude", "Moisture ( % )"]].to_string(index=False))


Total records on 23/04/2026: 6
          TimeStamp  Latitude  Longitude  Moisture ( % )
2026-04-23 05:14:00  -3.06246  -79.23457           54.09
2026-04-23 05:15:00  -3.06246  -79.23457           54.09
2026-04-23 05:17:00  -3.06253  -79.23454           56.52
2026-04-23 05:18:00  -3.06244  -79.23457           55.36
2026-04-23 05:22:00  -3.06255  -79.23467           55.15
2026-04-23 05:24:00  -3.06256  -79.23463           56.46


In [2]:
import math

# Compute bounding box ±100 m around all measurement points
lat_pts = df_day["Latitude"].values
lon_pts = df_day["Longitude"].values

# Degrees-per-metre conversion at mean latitude
mean_lat = lat_pts.mean()
delta_lat = 100 / 111_320                                  # ~0.000899°
delta_lon = 100 / (111_320 * math.cos(math.radians(mean_lat)))  # ~0.000900° at -3°

bbox = {
    "lat_min": lat_pts.min() - delta_lat,
    "lat_max": lat_pts.max() + delta_lat,
    "lon_min": lon_pts.min() - delta_lon,
    "lon_max": lon_pts.max() + delta_lon,
}

print("Bounding box (±100 m):")
for k, v in bbox.items():
    print(f"  {k}: {v:.6f}°")


Bounding box (±100 m):
  lat_min: -3.063458°
  lat_max: -3.061542°
  lon_min: -79.235570°
  lon_max: -79.233640°


In [4]:
smap_csv = r"soil_moisture\SMAP_0226_0423.csv"

df_smap = pd.read_csv(smap_csv, parse_dates=["date"])
df_smap = df_smap.drop(columns=[".geo"])

# Filter to 23/04/2026
df_smap_day = df_smap[df_smap["date"].dt.date == pd.Timestamp("2026-04-23").date()].copy()

print(f"SMAP records on 23/04/2026: {len(df_smap_day)}")
print(df_smap_day.to_string(index=False))

print(f"\nSMAP SM mean (23/04/2026) : {df_smap_day['soil_moisture'].mean():.4f} cm³/cm³")
print(f"Field SM mean (23/04/2026): {df_day['Moisture ( % )'].mean() / 100:.4f} cm³/cm³")


SMAP records on 23/04/2026: 0
Empty DataFrame
Columns: [system:index, date, soil_moisture]
Index: []

SMAP SM mean (23/04/2026) : nan cm³/cm³
Field SM mean (23/04/2026): 0.5528 cm³/cm³
